In [2]:
import sys
import os
import torch

In [3]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment

/home/brian_bosho/FP/FP/src/utils.py:58: SyntaxWarning: invalid escape sequence '\m'
  """


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
from run import load_and_split_with_khop, load_and_split_with_feature_prop
data, dataset, clients_data, test_data,  = load_and_split_with_khop("Cora", device, num_clients=10, beta=0.5, hop=1, imputation_method="adjacency")


In [5]:
# check client 0 data
print(clients_data[0])
# check number of nodes in client 0 with zerof feature vectors
# also check train/test/val nodes the number
#total nodes in client 0
total_nodes = clients_data[0].x.shape[0]
#nodes with zero feature vectors
zfv_nodes = torch.sum(clients_data[0].x == 0)
print(f"Number of nodes in client 0 with zero feature vectors: {torch.sum(clients_data[0].x == 0)}")
print(f"Number of train nodes in client 0: {torch.sum(clients_data[0].train_mask)}")
print(f"Number of test nodes in client 0: {torch.sum(clients_data[0].test_mask)}")
print(f"Number of val nodes in client 0: {torch.sum(clients_data[0].val_mask)}")
# chek how many nodes have labels
print(f"Number of nodes with labels in client 0: {torch.sum(clients_data[0].y != -100)}")



Data(x=[344, 1433], edge_index=[2, 1056], y=[344], train_mask=[344], val_mask=[344], test_mask=[344], mapping=[103])
Number of nodes in client 0 with zero feature vectors: 421820
Number of train nodes in client 0: 11
Number of test nodes in client 0: 34
Number of val nodes in client 0: 16
Number of nodes with labels in client 0: 103


In [6]:
# load with fp and check the same things
data, dataset, clients_data, test_data,  = load_and_split_with_feature_prop("ogbn-arxiv", device, num_clients=10, beta=0.5, hop=1)

# check client 0 data
print(clients_data[0])
# check number of nodes in client 0 with zerof feature vectors
print(f"Number of nodes in client 0 with zero feature vectors: {torch.sum(clients_data[0].x == 0)}")
print(f"Number of train nodes in client 0: {torch.sum(clients_data[0].train_mask)}")
print(f"Number of test nodes in client 0: {torch.sum(clients_data[0].test_mask)}")
print(f"Number of val nodes in client 0: {torch.sum(clients_data[0].val_mask)}")
    


/home/brian_bosho/miniconda3/lib/python3.12/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.lo

Data(x=[89795, 128], edge_index=[2, 1748783], y=[89795], train_mask=[89795], val_mask=[89795], test_mask=[89795], mapping=[19792])
Number of nodes in client 0 with zero feature vectors: 6
Number of train nodes in client 0: 10142
Number of test nodes in client 0: 5917
Number of val nodes in client 0: 3733


In [7]:
import torch

print(f"Number of GPUs available: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU available")

Number of GPUs available: 1
GPU 0: NVIDIA L40-48Q


In [8]:
# from federated_learning_cuda import load_configuration, main_experiment
from run import load_configuration, main_experiment
import os
import ray
ray.shutdown()

if __name__ == "__main__":
    # Define all options in a list
    data_loading_options = ["zero_hop", "diffusion"]
    model_types = ["GCN"]
        
    # Load configuration once since it's common for all runs
    clients_num, beta, cfg = load_configuration()

    clients_num = 10

    results_list = []

    # Main results directory
    main_dir = 'propagation_results'
    dataset_name = "Citeseer"  # You can change this if you have different datasets

    # Loop over all combinations of data_loading_option and model_type
    for data_loading_option in data_loading_options:
        for model_type in model_types:
            # Create a structured directory for each experiment
            result_name = f"{dataset_name}_{data_loading_option}_{model_type}"
            results_dir = os.path.join(main_dir, result_name)

            # Create the directory if it doesn't exist
            os.makedirs(results_dir, exist_ok=True)

            print(f"Running experiment with data_loading_option: {data_loading_option}, model_type: {model_type}")
            result = main_experiment(clients_num, beta, data_loading_option, model_type, cfg, dataset_name=dataset_name, hop=2)

            results_list.append(result)

            # results_data, result_text = result
            
           
            
            # # Create a unique filename for each combination
            # filename = f'results_{dataset_name}_{data_loading_option}_{model_type}.txt'
            # filepath = os.path.join(results_dir, filename)
            
            # # Write the result to the text file
            # with open(filepath, 'w') as f:
            #     f.write(result_with_newlines)
            
            # print(f"Results saved to {filepath}\n")

Running experiment with data_loading_option: zero_hop, model_type: GCN
DEVICE: cuda
Data loading option is zero_hop
Model type is GCN


2025-04-28 19:37:29,683	INFO worker.py:1816 -- Started a local Ray instance.


data_loading_option: zero_hop
Data loaded
Device is cuda
Citeseer()
10


(FLClient pid=2930003) 2025-04-28 19:37:52,112 - INFO - Epoch   0| Train Loss: 1.778| Train Accuracy: 0.105
(FLClient pid=2930003) 2025-04-28 19:37:52,381 - INFO - Epoch   1| Train Loss: 1.559| Train Accuracy: 0.474
(FLClient pid=2930003) 2025-04-28 19:37:52,407 - INFO - Epoch   1| Validation Loss: 1.822, Validation Accuracy: 0.104
(FLClient pid=2930021) 2025-04-28 19:37:58,179 - INFO - Epoch   1| Train Loss: 1.602| Train Accuracy: 0.500 [repeated 4x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(FLClient pid=2930021) 2025-04-28 19:37:58,183 - INFO - Epoch   1| Validation Loss: 1.730, Validation Accuracy: 0.260 [repeated 2x across cluster]
(FLClient pid=2930013) 2025-04-28 19:38:03,325 - INFO - Epoch   0| Train Loss: 1.788| Train Accuracy: 0.083 [repeated 13x across cluster]
(FLClient pid=2930019) 2025-04-

Training done
Round 1
Train Loss: 1.778, Train Accuracy: 0.714
Train Loss: 1.764, Train Accuracy: 0.538
Train Loss: 1.730, Train Accuracy: 0.500
Train Loss: 1.736, Train Accuracy: 0.667
Train Loss: 1.795, Train Accuracy: 0.667
Train Loss: 1.788, Train Accuracy: 0.444
Train Loss: 1.822, Train Accuracy: 0.474
Train Loss: 1.761, Train Accuracy: 0.571
Train Loss: 1.793, Train Accuracy: 0.417
Train Loss: 1.762, Train Accuracy: 0.467
Round 2
Train Loss: 1.773, Train Accuracy: 0.714
Train Loss: 1.760, Train Accuracy: 0.462
Train Loss: 1.691, Train Accuracy: 0.700
Train Loss: 1.693, Train Accuracy: 0.750
Train Loss: 1.783, Train Accuracy: 0.556
Train Loss: 1.770, Train Accuracy: 0.556
Train Loss: 1.792, Train Accuracy: 0.421
Train Loss: 1.739, Train Accuracy: 0.643
Train Loss: 1.735, Train Accuracy: 0.583
Train Loss: 1.765, Train Accuracy: 0.800
Validation Loss: 1.739, Validation Accuracy: 0.373
Validation Loss: 1.744, Validation Accuracy: 0.267
Validation Loss: 1.745, Validation Accuracy: 0.2

...

(FLClient pid=2933261) 2025-04-28 19:39:55,964 - INFO - Epoch   1| Train Loss: 1.532| Train Accuracy: 0.857 [repeated 26x across cluster]
(FLClient pid=2933261) 2025-04-28 19:39:55,965 - INFO - Epoch   1| Validation Loss: 1.783, Validation Accuracy: 0.186 [repeated 13x across cluster]
(FLClient pid=2933254) 2025-04-28 19:40:01,076 - INFO - Epoch   0| Train Loss: 1.783| Train Accuracy: 0.158 [repeated 7x across cluster]
(FLClient pid=2933244) 2025-04-28 19:40:00,729 - INFO - Epoch   1| Validation Loss: 1.817, Validation Accuracy: 0.053 [repeated 3x across cluster]


Training done
Round 1
Train Loss: 1.783, Train Accuracy: 0.857
Train Loss: 1.786, Train Accuracy: 0.538
Train Loss: 1.738, Train Accuracy: 0.700
Train Loss: 1.739, Train Accuracy: 0.500
Train Loss: 1.817, Train Accuracy: 0.556
Train Loss: 1.833, Train Accuracy: 0.778
Train Loss: 1.839, Train Accuracy: 0.474
Train Loss: 1.767, Train Accuracy: 0.571
Train Loss: 1.758, Train Accuracy: 0.500
Train Loss: 1.727, Train Accuracy: 0.467
Round 2
Train Loss: 1.762, Train Accuracy: 0.857
Train Loss: 1.770, Train Accuracy: 0.538
Train Loss: 1.710, Train Accuracy: 0.600
Train Loss: 1.687, Train Accuracy: 0.917
Train Loss: 1.802, Train Accuracy: 0.667
Train Loss: 1.779, Train Accuracy: 0.667
Train Loss: 1.812, Train Accuracy: 0.474
Train Loss: 1.732, Train Accuracy: 0.714
Train Loss: 1.738, Train Accuracy: 0.750
Train Loss: 1.694, Train Accuracy: 0.667
Validation Loss: 1.717, Validation Accuracy: 0.407
Validation Loss: 1.740, Validation Accuracy: 0.267
Validation Loss: 1.728, Validation Accuracy: 0.3

(FLClient pid=2933266) 2025-04-28 19:40:14,004 - INFO - Epoch   1| Train Loss: 1.629| Train Accuracy: 0.571 [repeated 33x across cluster]
(FLClient pid=2933266) 2025-04-28 19:40:14,006 - INFO - Epoch   1| Validation Loss: 1.776, Validation Accuracy: 0.153 [repeated 17x across cluster]
(FLClient pid=2933264) 2025-04-28 19:40:19,173 - INFO - Epoch   1| Train Loss: 1.493| Train Accuracy: 0.600 [repeated 6x across cluster]
(FLClient pid=2933264) 2025-04-28 19:40:19,175 - INFO - Epoch   1| Validation Loss: 1.704, Validation Accuracy: 0.340 [repeated 3x across cluster]


Training done
Round 1
Train Loss: 1.776, Train Accuracy: 0.571
Train Loss: 1.805, Train Accuracy: 0.538
Train Loss: 1.704, Train Accuracy: 0.600
Train Loss: 1.706, Train Accuracy: 0.250
Train Loss: 1.803, Train Accuracy: 0.778
Train Loss: 1.814, Train Accuracy: 0.667
Train Loss: 1.850, Train Accuracy: 0.368
Train Loss: 1.740, Train Accuracy: 0.643
Train Loss: 1.795, Train Accuracy: 0.500
Train Loss: 1.730, Train Accuracy: 0.600
Round 2
Train Loss: 1.740, Train Accuracy: 1.000
Train Loss: 1.765, Train Accuracy: 0.692
Train Loss: 1.658, Train Accuracy: 0.700
Train Loss: 1.661, Train Accuracy: 0.833
Train Loss: 1.767, Train Accuracy: 0.889
Train Loss: 1.827, Train Accuracy: 0.667
Train Loss: 1.808, Train Accuracy: 0.789
Train Loss: 1.722, Train Accuracy: 0.643
Train Loss: 1.761, Train Accuracy: 0.667
Train Loss: 1.699, Train Accuracy: 0.667
Validation Loss: 1.711, Validation Accuracy: 0.339
Validation Loss: 1.735, Validation Accuracy: 0.289
Validation Loss: 1.701, Validation Accuracy: 0.3

(FLClient pid=2935430) 2025-04-28 19:40:21,766 - INFO - Epoch   1| Train Loss: 1.560| Train Accuracy: 0.667 [repeated 32x across cluster]
(FLClient pid=2935430) 2025-04-28 19:40:21,790 - INFO - Epoch   1| Validation Loss: 1.699, Validation Accuracy: 0.333 [repeated 16x across cluster]


In [11]:
result_0 = results_list[0]

In [12]:
result_0

({'experiment_config': {'device': 'cuda',
   'data_loading_option': 'zero_hop',
   'model_type': 'GCN',
   'dataset': 'Citeseer',
   'num_clients': 10,
   'beta': 10000,
   'hop': 2,
   'fulltraining_flag': False},
  'rounds': [{'round': 1,
    'global_result': 0.672,
    'client_result': 0.6017525987587816},
   {'round': 2, 'global_result': 0.673, 'client_result': 0.6215892191602114},
   {'round': 3, 'global_result': 0.681, 'client_result': 0.613751449480364}],
  'summary': {'global_results': [0.672, 0.673, 0.681],
   'client_results': [0.6017525987587816,
    0.6215892191602114,
    0.613751449480364],
   'average_global_result': 0.6753333333333335,
   'average_client_result': 0.6123644224664523,
   'std_global': 0.004027681991198194,
   'std_client': 0.008157440782933302}},
 'DEVICE: cuda\nData loading option is zero_hop\nModel type is GCN\nFull training flag is False\n\nFinal Results:\nThe global test results: [0.672, 0.673, 0.681]\nThe client test results: [0.6017525987587816, 0.6

In [13]:
result_1 = results_list[1]
result_1



({'experiment_config': {'device': 'cuda',
   'data_loading_option': 'diffusion',
   'model_type': 'GCN',
   'dataset': 'Citeseer',
   'num_clients': 10,
   'beta': 10000,
   'hop': 2,
   'fulltraining_flag': False},
  'rounds': [{'round': 1,
    'global_result': 0.675,
    'client_result': 0.6448521090414241},
   {'round': 2, 'global_result': 0.678, 'client_result': 0.643967586062272},
   {'round': 3, 'global_result': 0.687, 'client_result': 0.6531682647791668}],
  'summary': {'global_results': [0.675, 0.678, 0.687],
   'client_results': [0.6448521090414241,
    0.643967586062272,
    0.6531682647791668],
   'average_global_result': 0.68,
   'average_client_result': 0.6473293199609543,
   'std_global': 0.00509901951359279,
   'std_client': 0.00414451868277125}},
 'DEVICE: cuda\nData loading option is diffusion\nModel type is GCN\nFull training flag is False\n\nFinal Results:\nThe global test results: [0.675, 0.678, 0.687]\nThe client test results: [0.6448521090414241, 0.643967586062272

### Tests